# 🔑 Solutions · Lesson 07: Deep-Learning Forecasting: RNN, LSTM, GRU & Transformers

> Try every problem yourself first! Peek at hints before peeking here. Run the setup cell, then read each solution **and its explanation**: there is often more than one valid way.

In [1]:
import sys; sys.path.append("..")            # lets us import course_utils.py from the parent folder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.tools.sm_exceptions         # (imported first: statsmodels registers its own warning filters on import)
import warnings; warnings.filterwarnings("ignore")   # keep the notebook readable; in real projects READ your warnings!
from course_utils import *                    # data loaders, metrics, plotting helpers, check()

In [2]:
from dl_utils import *
mz = load_weekly("maize")

### ✅ Solution 7.1 · Count the windows

**Problem recap:** (a) A series has `T = 300` observations. With input length `L = 26` and horizon `H = 4`, how many training windows can you cut (no warm-up NaNs)?
  (b) Now use the real code: build windows for **rice** with `make_windows(sf, lp, L=52, H=8)` and report `N`. Store (b) in `n_windows` and print the shape of `X`.

In [3]:
rice = load_weekly('rice')
print('(a) windows:', 300 - 26 - 4 + 1)
sf, lp = make_step_features(rice)
X, Y, org = make_windows(sf, lp, 52, 8)
print('X shape (N, L, F):', X.shape, '| Y shape (N, H):', Y.shape)
n_windows = len(X)
check('windows for rice (L=52, H=8)', n_windows, 556)

(a) windows: 271
X shape (N, L, F): (556, 52, 9) | Y shape (N, H): (556, 8)
✅ windows for rice (L=52, H=8): correct!


**Why this works:** (a) 300 − 26 − 4 + 1 = 271. For the real data a few weeks are lost at the start because rolling features (12-week mean) need warm-up, giving 556 windows: a *tiny* dataset by deep-learning standards, which is why we keep the networks small.

### ✅ Solution 7.2 · Count parameters before you run

**Problem recap:** A recurrent layer with input size `F` and hidden size `H` has `H(F+H)+2H` weights per 'gate block'. RNN has **1** block, GRU **3**, LSTM **4**; add a linear head `H×8+8`.
  Predict (on paper!) the total trainable parameters of RNN, GRU and LSTM for `F = 9`, `H = 64`, then verify with `count_params(build_model(...))`.
  Store the LSTM count in `lstm_params`.

In [4]:
counts = {k: count_params(build_model(k, 9, 8, 52, 64)) for k in ['rnn', 'gru', 'lstm']}
print(counts, '| predicted:', {'rnn': 4800 + 520, 'gru': 3 * 4800 + 520, 'lstm': 4 * 4800 + 520})
lstm_params = counts['lstm']
check('LSTM parameters', lstm_params, 19720)

{'rnn': 5320, 'gru': 14920, 'lstm': 19720} | predicted: {'rnn': 5320, 'gru': 14920, 'lstm': 19720}
✅ LSTM parameters: correct!


**Why this works:** An LSTM has four internal transformations (input, forget, output gate and candidate), a GRU three (reset, update, candidate), a plain RNN one. More gates ⇒ more parameters ⇒ more expressive but also more prone to over-fit 556 windows.

### ✅ Solution 7.3 · Tune a GRU the honest way

**Problem recap:** On **onion** use `evaluate_split(onion, 'gru', h=4, hidden=..., dropout=...)` for `hidden ∈ {16, 32, 64}` and `dropout ∈ {0.0, 0.2}`.
  Choose the configuration with the lowest **validation loss** (`min(result['best_val'])`), *not* the lowest test MAE, then report its test MAE and the naive MAE.
  Why is picking by test MAE a mistake?

In [5]:
import itertools
onion = load_weekly('onion')
rows = []
for hidden, dropout in itertools.product([16, 32, 64], [0.0, 0.2]):
    r = evaluate_split(onion, 'gru', h=4, hidden=hidden, dropout=dropout)
    rows.append(dict(hidden=hidden, dropout=dropout, val_loss=r['best_val'][0], test_MAE=r['per_seed_mae'][0], naive_MAE=r['naive_mae']))
t = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
print(t.round(3))
print('Chosen by validation loss ->', t.iloc[0][['hidden', 'dropout']].to_dict(), '| its test MAE:', round(t.iloc[0]['test_MAE'], 3))

   hidden  dropout  val_loss  test_MAE  naive_MAE
0      16      0.0     0.181     4.489      7.162
1      64      0.0     0.187     4.555      7.162
2      64      0.2     0.192     4.658      7.162
3      16      0.2     0.198     4.561      7.162
4      32      0.2     0.215     5.945      7.162
5      32      0.0     0.249     6.421      7.162
Chosen by validation loss -> {'hidden': 16.0, 'dropout': 0.0} | its test MAE: 4.489


**Why this works:** The validation slice (most recent training windows) is the *only* thing allowed to influence model choice. The test MAE column is just for reporting afterwards. Compare the two orderings: the top pick by validation loss happens to be good on the test set here, but the order of the remaining configurations is *not* the same, and with a different seed or split it can easily differ at the top too. That is exactly why peeking at test results would mislead you.

### ✅ Solution 7.4 · How noisy is a neural network? Seeds and ensembles

**Problem recap:** Train an **LSTM** on **maize** with seeds `0, 1, 2, 3, 4` via `evaluate_split(mz, 'lstm', h=8, seeds=(0, 1, 2, 3, 4))`. Report the mean and standard deviation of the per-seed test MAE,
  the MAE of the **averaged** (ensemble) forecast, and the naive MAE. What does the spread tell you about trusting a single run?

In [6]:
mz = load_weekly('maize')
r = evaluate_split(mz, 'lstm', h=8, seeds=(0, 1, 2, 3, 4))
m = np.array(r['per_seed_mae'])
print('per-seed MAE:', m.round(2))
print(f'mean ± std : {m.mean():.2f} ± {m.std():.2f}')
print(f'ensemble   : {r["ens_mae"]:.2f}   | naive: {r["naive_mae"]:.2f}')

per-seed MAE: [7.92 7.98 8.24 8.03 7.59]
mean ± std : 7.95 ± 0.21
ensemble   : 7.24   | naive: 14.22


**Why this works:** Same data, same code, different random seed: different answers. A single training run is *one draw* from a distribution; averaging several seeds usually beats the average single run and is far more stable. Whenever you compare neural nets, compare **distributions over seeds**, not single runs.

### ✅ Solution 7.5 · How much history does the network need?

**Problem recap:** For a **GRU** on **maize** (h = 8, `seeds=(0, 1, 2)`), vary the input window length `L ∈ {13, 26, 52, 78}` using `evaluate_split(..., L=L)`.
  Compare the ensemble MAE and the naive MAE for each `L` (the *test targets are identical across L*, so the comparison is fair; why?). Is longer always better? Relate your finding to the 52-week seasonal cycle.

In [7]:
mz = load_weekly('maize')
out = {}
for L in [13, 26, 52, 78]:
    r = evaluate_split(mz, 'gru', h=8, L=L, seeds=(0, 1, 2))
    out[L] = dict(ensemble_MAE=r['ens_mae'], naive_MAE=r['naive_mae'], n_test=r['n_test'])
pd.DataFrame(out).T.rename_axis('window length L').round(2)

,ensemble_MAE,naive_MAE,n_test
window length L,,,
13,7.09,14.22,148.0
26,7.04,14.22,148.0
52,6.72,14.22,148.0
78,7.96,14.22,148.0


**Why this works:** Because every test origin is later than week 469 and `L` only controls how much *history* each window shows, all four settings are scored on exactly the same targets. More history gives more evidence about the seasonal position and lagged drivers, but also more input values to over-fit with only ~500 windows, so there is usually a sweet spot rather than 'longer is always better'. In our run a full 52-week window (one complete seasonal cycle) did best, while 78 weeks was clearly worse.